In [ ]:
import pandas as pd
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from pinecone import Pinecone, ServerlessSpec


c:\python notebooks\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("../data/movie_reviews/IMDB_Dataset.csv")

def clean_text(text):
    text = re.sub(r'<.*?>', '', text)
    return text.lower()

df['review'] = df['review'].apply(clean_text)

df['sentiment'] = df['sentiment'].map({'positive':1, 'negative':0})

print(df.head())


NameError: name 'pd' is not defined

In [8]:
# Replace with your API key
pc = Pinecone(api_key="pcsk_UrePP_56vSCDDph7vqheXLnBZzXn8gRVSmZbwpU8YWA559S8UXDsjFGv7tQ9UCofYD7hv")

index_name = "imdb-sentiment"

# Create index (run only once)
pc.create_index(
    name=index_name,
    dimension=384,  # embedding size of MiniLM
    metric="cosine",
    spec=ServerlessSpec(
        cloud="aws",
        region="us-east-1"
    )
)

index = pc.Index(index_name)


In [9]:
model = SentenceTransformer('all-MiniLM-L6-v2')

texts = df['review'].tolist()
embeddings = model.encode(texts, show_progress_bar=True)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 935.76it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 1563/1563 [13:27<00:00,  1.94it/s]


In [ ]:
start = 0
end = 50000
batch_size = 100

for i in range(start, end, batch_size):
    batch = []
    
    for j in range(i, min(i + batch_size, end)):
        batch.append(
            (
                str(j),   # ID continues properly
                embeddings[j].tolist(),
                {"sentiment": int(df['sentiment'][j])}
            )
        )
    
    index.upsert(vectors=batch)
    print(f"Uploaded {i} to {min(i+batch_size, end)}")


Uploaded 10000 to 10100
Uploaded 10100 to 10200
Uploaded 10200 to 10300
Uploaded 10300 to 10400
Uploaded 10400 to 10500
Uploaded 10500 to 10600
Uploaded 10600 to 10700
Uploaded 10700 to 10800
Uploaded 10800 to 10900
Uploaded 10900 to 11000
Uploaded 11000 to 11100
Uploaded 11100 to 11200
Uploaded 11200 to 11300
Uploaded 11300 to 11400
Uploaded 11400 to 11500
Uploaded 11500 to 11600
Uploaded 11600 to 11700
Uploaded 11700 to 11800
Uploaded 11800 to 11900
Uploaded 11900 to 12000
Uploaded 12000 to 12100
Uploaded 12100 to 12200
Uploaded 12200 to 12300
Uploaded 12300 to 12400
Uploaded 12400 to 12500
Uploaded 12500 to 12600
Uploaded 12600 to 12700
Uploaded 12700 to 12800
Uploaded 12800 to 12900
Uploaded 12900 to 13000
Uploaded 13000 to 13100
Uploaded 13100 to 13200
Uploaded 13200 to 13300
Uploaded 13300 to 13400
Uploaded 13400 to 13500
Uploaded 13500 to 13600
Uploaded 13600 to 13700
Uploaded 13700 to 13800
Uploaded 13800 to 13900
Uploaded 13900 to 14000
Uploaded 14000 to 14100
Uploaded 14100 t

In [19]:
index.describe_index_stats()


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '189',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 16 Feb 2026 16:58:22 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '40',
                                    'x-pinecone-request-latency-ms': '39',
                                    'x-pinecone-response-duration-ms': '41'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 50000}},
 'storageFullness': 0.0,
 'total_vector_count': 50000,
 'vector_type': 'dense'}

In [20]:
def predict_sentiment(text):
    text = clean_text(text)
    embedding = model.encode(text).tolist()
    
    result = index.query(
        vector=embedding,
        top_k=5,
        include_metadata=True
    )
    
    sentiments = [match['metadata']['sentiment'] for match in result['matches']]
    
    prediction = round(np.mean(sentiments))
    
    return "Positive" if prediction == 1 else "Negative"


In [1]:
print(predict_sentiment("The movie was boring and too slow"))
print(predict_sentiment("Absolutely amazing and inspiring film"))


NameError: name 'predict_sentiment' is not defined

In [22]:
import numpy as np

correct = 0
total = 1000   # test on 1000 reviews for faster evaluation

start_test = 10000

for i in range(start_test, start_test + total):
    
    review_text = df['review'][i]
    true_label = int(df['sentiment'][i])
    
    # Create embedding
    embedding = model.encode(review_text).tolist()
    
    # Query Pinecone
    result = index.query(
        vector=embedding,
        top_k=5,
        include_metadata=True
    )
    
    # Get neighbor sentiments
    neighbor_sentiments = [
        match['metadata']['sentiment']
        for match in result['matches']
    ]
    
    # Majority vote
    prediction = round(np.mean(neighbor_sentiments))
    
    if prediction == true_label:
        correct += 1

accuracy = correct / total

print("Pinecone Vector Accuracy:", accuracy)


Pinecone Vector Accuracy: 0.877


In [25]:
np.save("../artifacts/imdb_embeddings.npy", embeddings)
